In [1]:
import pandas as pd
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
import re
# import io
import os
from tqdm import tqdm # Barra de progresso
import matplotlib.pyplot as plt # Para analise de resultados 
import seaborn as sns # Para analise de resultados
from concurrent.futures import ThreadPoolExecutor

In [2]:
# ========================================================================================
# criacao do mapa de diagnosticos para CRs a partir do arquivo diagnosticos_topografia.txt
# ========================================================================================

CID_RE = re.compile(r"\b([A-TV-Z]\d{2}(?:\.\d{1,2})?)\b")

def extract_cids(text: str) -> set[str]:
    if not text:
        return set()
    cids = set(m.group(1).upper().replace(".", "") for m in CID_RE.finditer(text))
    return cids

# Mapeamento dos cabeçalhos do arquivo de texto para os CRs
header_to_cr = {
    'pulmao_maligno': 'Tumores de Pulmão e Tórax',
    'timo_maligno': 'Tumores de Pulmão e Tórax',
    'pleura_pericardio_maligno': 'Tumores de Pulmão e Tórax',
    'coração_maligno': 'Tumores de Pulmão e Tórax',
    'nasais_paranasais_base_craniana_maligno': 'Tumores de cabeça e pescoço',
    'nasofaringe_maligno': 'Tumores de cabeça e pescoço',
    'hipofaringe_laringe_parafaringe_maligno': 'Tumores de cabeça e pescoço',
    'glândulas_salivares_maligno': 'Tumores de cabeça e pescoço',
    'tireoide': 'Tumores de cabeça e pescoço', 
    'tireoide_maligno': 'Tumores de cabeça e pescoço',
    'paratireoide_maligno': 'Tumores de cabeça e pescoço',
    'esofago_maligno': 'Tumores do Aparelho Digestivo Alto',
    'estomago_maligno': 'Tumores do Aparelho Digestivo Alto',
    'intestino_delgado_ampula_maligno': 'Tumores do Aparelho Digestivo Alto',
    'figado_maligno': 'Tumores do Aparelho Digestivo Alto',
    'via_biliar_maligno': 'Tumores do Aparelho Digestivo Alto',
    'vesicula_biliar_maligno': 'Tumores do Aparelho Digestivo Alto',
    'pancreas_maligno': 'Tumores do Aparelho Digestivo Alto',
    'colon_reto_maligno': 'Tumores colorretais',
    'canal_anal_maligno': 'Tumores colorretais',
    'apendice': 'Tumores colorretais',
    'adrenal_maligno': 'Tumores Urológicos', # Adrenal frequentemente tratada pela uro
    'rim_maligno': 'Tumores Urológicos', # Assumido se houver seção específica, ou via regra geral
    'bexiga_maligno': 'Tumores Urológicos', # Idem
    'prostata_maligno': 'Tumores Urológicos', # Idem
    'penis_maligno': 'Tumores Urológicos', # Idem
    'testiculo_maligno': 'Tumores Urológicos', # Idem
    'mama_maligno': 'Tumores de Mama', # Idem
    'ovário_maligno': 'Tumores Ginecológicos', 
    'Corpo_do_útero_maligno': 'Tumores Ginecológicos', 
    'Colo_uterino_maligno': 'Tumores Ginecológicos',
    'ginecologico': 'Tumores Ginecológicos', 
    'SNC': 'Tumores do Sistema Nervoso Central',
    'pele_maligno': 'Tumores cutâneos',
    'pulmao_neuroendocrino': 'Tumores neuroendócrinos',
    'timo_neuroendocrino': 'Tumores neuroendócrinos',
    'pancreas_neuroendocrino': 'Tumores neuroendócrinos',
    'neuroendocrino_orgaos_nao_endocrinos': 'Tumores neuroendócrinos',
    'adrenal_medula_paraganglios_neuroendocrino': 'Tumores neuroendócrinos',
    'hematolinfoide': 'Neoplasias hematológicas',
    'hematolinfoide_Neoplasias_mieloproliferativas': 'Neoplasias hematológicas',
    'hematolinfoide_Proliferações_e_neoplasias_mieloides': 'Neoplasias hematológicas',
    'hematolinfoide_Leucemia_mieloide_aguda': 'Neoplasias hematológicas',
    'adipocitico_partes_moles': 'Sarcomas e tumores ósseos',
    'fibroblastica_partes_moles': 'Sarcomas e tumores ósseos',
    'vascular_partes_moles': 'Sarcomas e tumores ósseos', 
    'tumores_mesenquimatosos_pulmão': 'Tumores de Pulmão e Tórax', 
    'trato_urinario_maligno': 'Tumores Urológicos',
}

# Carregar base de conhecimento (simulado a partir do conteúdo lido)
knowledge_base_content = """
[O conteúdo do arquivo diagnosticos_topografia.txt foi processado internamente para criar o mapa]
"""

# Função simplificada para carregar o mapa de diagnósticos a partir do arquivo txt (lógica de parsing)
def parse_knowledge_base_file(filepath):
    mapping = {}
    
    # Mapeamentos manuais extras para garantir precisão
    manual_map = {
        'carcinoma basocelular': 'Tumores cutâneos',
        'carcinoma espinocelular': 'Tumores cutâneos', 
        'melanoma': 'Tumores cutâneos',
        'adenocarcinoma de prostata': 'Tumores Urológicos',
        'adenocarcinoma acinar usual': 'Tumores Urológicos',
        'carcinoma ductal invasivo': 'Tumores de Mama',
        'carcinoma lobular invasivo': 'Tumores de Mama',
    }
    mapping.update(manual_map)

    # --- LÓGICA PARA EXCEL (.xlsx) ---
    if filepath.endswith('.xlsx'):
        try:
            # Lê o Excel. Assume-se que a 1ª coluna é o Diagnóstico e a 2ª é o Grupo (CR)
            # Se o excel não tiver cabeçalho, adicione header=None no read_excel
            df = pd.read_excel(filepath)
            
            # Itera sobre as linhas do Excel
            for index, row in df.iterrows():
                # Pega valores da primeira (0) e segunda (1) coluna
                term = str(row.iloc[1]).strip().lower() 
                group = str(row.iloc[2]).strip()
                
                # Evita adicionar linhas vazias ou inválidas
                if term and group and term != 'nan' and group != 'nan':
                    mapping[term] = group
                    
            print(f"Sucesso: Carregados {len(df)} registros do Excel.")
            
        except Exception as e:
            print(f"Erro ao ler arquivo Excel: {e}")

        # ==========================================
    # LÓGICA PARA CSV (.csv) - NOVO!
    # ==========================================
    elif filepath.endswith('.csv'):
        try:
            # Tenta ler com separador vírgula (padrão) ou ponto-e-vírgula (comum no Brasil)
            # O 'low_memory=False' ajuda se o arquivo for muito grande misturando tipos
            try:
                df = pd.read_csv(filepath)
            except:
                # Fallback: Se der erro, tenta com ponto e vírgula
                df = pd.read_csv(filepath, sep=';')

            for index, row in df.iterrows():
                # 1. Pega o ALVO (Classificação Final) - Coluna F (Índice 5)
                # Verifica se a célula existe antes de converter para string
                raw_group = row.iloc[5]
                
                # Se a classificação estiver vazia (NaN), PULA essa linha (caso das linhas 4-11 da imagem)
                if pd.isna(raw_group) or str(raw_group).strip() == '':
                    continue

                group = str(raw_group).strip()
                if group.startswith("CR "): group = group[3:].strip()
                
                # Se chegou aqui, temos um grupo válido. Vamos mapear TODAS as colunas úteis.
                
                # Coluna A: SNOMED (ex: M-85503)
                snomed = str(row.iloc[0]).strip().lower()
                if snomed and snomed != 'nan': mapping[snomed] = group

                # Coluna B: Label em Inglês (ex: Acinar cell carcinoma)
                label = str(row.iloc[1]).strip().lower()
                if label and label != 'nan': mapping[label] = group

                # Coluna D: CID (ex: C34)
                cid = str(row.iloc[3]).strip().lower()
                if cid and cid != 'nan': mapping[cid] = group

                # Coluna E: Descrição CID (ex: Neoplasia maligna...)
                desc_cid = str(row.iloc[4]).strip().lower()
                if desc_cid and desc_cid != 'nan': mapping[desc_cid] = group

            print(f"CSV processado: Mapa agora tem {len(mapping)} termos totais.")
            
        except Exception as e:
            print(f"Erro ao ler arquivo CSV: {e}")

    # Lê o arquivo real
    else:
        try:
            
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()

            lines = content.split('\n')
            current_cr = None
            
            for line in lines:
                line = line.strip()
                if line.startswith('###'):
                    section = line.replace('###', '').strip().lower()
                    
                    # Tenta achar o CR pelo dicionário header_to_cr
                    current_cr = None
                    for key in header_to_cr:
                        if key in section:
                            current_cr = header_to_cr[key]
                            break
                    
                    # Correções (se o nome da seção mudou no txt)
                    if not current_cr:
                        if 'mama' in section: current_cr = 'Tumores de Mama'
                        elif 'utero' in section or 'ovario' in section: current_cr = 'Tumores Ginecológicos'
                        elif 'rim' in section or 'bexiga' in section or 'prostata' in section: current_cr = 'Tumores Urológicos'
                        elif 'snc' in section: current_cr = 'Tumores do Sistema Nervoso Central'
                        elif 'linfoma' in section or 'leucemia' in section: current_cr = 'Neoplasias hematológicas'
                        elif 'leucemia' in section or 'mieloma' in section: current_cr = 'Neoplasias hematológicas'
                        elif 'hematolinfoide' in section or 'Leucemia' in section: current_cr = 'Neoplasias hematológicas'
                        elif 'Proliferação Linfoide Atípica' in section or 'linfoide' in section: current_cr = 'Neoplasias hematológicas'

                elif line.startswith('-') and len(line) > 1:
                    diag = line[1:].strip().lower()
                    if current_cr:
                        mapping[diag] = current_cr
        except Exception as e:
            print(f"Erro ao ler arquivo TXT: {e}")
                
    return mapping



# ========================================================================================
# # Lista de termos benignos comuns para verificação rápida (além do arquivo)
# ========================================================================================

common_benign_terms = [
    "livre de neoplasia", "ausência de neoplasia", "Negativo Para Células Neoplásicas", "ausência de malignidade",
    "inflamação", "inflamatorio", "gastrite", "colite", "duodenite", "esofagite",
    "lipoma", "leiomioma", "nevo", "cisto", "fibrose", "Fibrose", "hiperplasia", "Hiperplasia", "polipo",
    "adenoma", "bocio", "bócio", "fibroadenoma", "papiloma", "queratose", "ceratose",
    "hemangioma", "linfangioma", "granuloma", "material insuficiente", "negativo para",
    "sem alterações", "dentro dos limites", "benigno", "corpo estranho", "ectasia", "Glomerulosclerose", "Miofibroblástico", 
    "Inflamatório"
]

def classify_case(diagnosis, topography, snomed, malignant_map_rules):
    """
    Classificação híbrida atualizada com regras de comportamento SNOMED.
    """
    diag_lower = str(diagnosis).lower()
    topo_lower = str(topography).lower()
    snomed_str = str(snomed).strip()
    
# ========================================================================================
# --- REGRAS SNOMED ---
# Verifica se o campo SNOMED não está vazio e tem formato válido (ex: M-80003)
# # ========================================================================================
    
    if snomed_str and len(snomed_str) > 0 and snomed_str[-1].isdigit():
        last_digit = snomed_str[-1]
        
        # Regra 0: Benigno
        if last_digit == '0':
            return "BENIGNO"
            
        # Regra 6: Metástase
        if last_digit == '6':
            return "METÁSTASE"
        
        # Regra 2: In Situ
        if last_digit == '2':
            return "IN SITU"
        
        # Final 1: A Zona Cinzenta
        if last_digit == '1':
            # Se for código 1 (incerto), mas o texto disser explicitamente os termos abaixo:
            # assumimos BENIGNO para fins de triagem de CR.
            if "hiperplasia" in diag_lower: return "BENIGNO"
            if "teratoma" in diag_lower: return "BENIGNO"
            if "adenoma" in diag_lower: return "BENIGNO"
            if "Negativo Para Células Neoplásicas" in diag_lower: return "BENIGNO"
                        
        # Regra 1: Borderline (Incerteza)
        # Retornamos INDET para que a IA decida baseada no Procedimento
        if last_digit == '1':
            return "INDET"
            
# ========================================================================================
# --- REGRAS TEXTUAIS (Continuação para códigos 3 ou vazios) ---
# ========================================================================================
    # --- REGRA 2: BENIGNO POR TEXTO (COM TRAVA DE SEGURANÇA) ---
    # Verifica se existe algum termo da lista 'common_benign_terms' no diagnóstico
    if any(term in diag_lower for term in common_benign_terms):
        
        # TRAVA DE SEGURANÇA:
        # Palavras que indicam câncer. Se alguma destas aparecer, a regra de benigno é CANCELADA.
        # Isso evita que "Adenocarcinoma" seja classificado como benigno por conter "Adenoma".
        palavras_proibidas = [
            "carcinoma", 
            "sarcoma", 
            "linfoma", 
            "melanoma", 
            "leucemia", 
            "maligno", 
            "maligna", 
            "neoplasia maligna", 
            "tumor maligno",
            "infiltrante",
            "invasivo",
            "seminoma",
            "mesotelioma"
        ]
        
        # Verifica se alguma palavra proibida está presente no texto
        eh_cancer_oculto = any(p in diag_lower for p in palavras_proibidas)
        
        # Só retornamos BENIGNO se encontrar um termo benigno E NÃO encontrar quyalquer termo maligno.
        if not eh_cancer_oculto:
             return "BENIGNO"
    
    # REGRA: Neuroendócrino
    if "neuroendócrino" in diag_lower or "neuroendocrino" in diag_lower or "Neuroendócrina" in diag_lower:
        return "Tumores neuroendócrinos"

    # REGRA: Sarcomas (Osso / Partes Moles)
    if "sarcoma" in diag_lower:
        return "Sarcomas e tumores ósseos"
    if "osso" in topo_lower:
        return "Sarcomas e tumores ósseos"
    if ("partes_moles" in topo_lower or "partes moles" in topo_lower) and "pele" not in topo_lower:
        return "Sarcomas e tumores ósseos"

    # REGRA: Pele (Topografia Específica)
    # Só aplica se NÃO for Sarcoma (já verificado acima)
    if "região frontal" in topo_lower or "regiao frontal" in topo_lower:
        return "Tumores cutâneos"
    if topo_lower.startswith("pele"):
        return "Tumores cutâneos"
    
    # Concatenando diagnóstico e topografia para buscar a palavra-chave em ambos
    # Isso resolve casos onde a topografia é vaga mas o diagnóstico diz o órgão
    texto_completo = (str(diagnosis) + " " + str(topography)).lower()

    # GRUPO 1: Órgãos que não se confundem facilmente
    
    # Urológicos (Resolve o problema da Próstata ser classificada como reto)
    if "prostata" in texto_completo or "próstata" in texto_completo: return "Tumores Urológicos"
    if "testiculo" in texto_completo or "testículo" in texto_completo: return "Tumores Urológicos"
    if "rim" in texto_completo: return "Tumores Urológicos"
    if "bexiga" in texto_completo or "ureter" in texto_completo: return "Tumores Urológicos"
    if "penis" in texto_completo or "pênis" in texto_completo: return "Tumores Urológicos"
    if "uretral" in texto_completo or "ureteral" in texto_completo: return "Tumores Urológicos"
    if "transuretral" in texto_completo: return "Tumores Urológicos"
    if "nefrectomia" in texto_completo: return "Tumores Urológicos"

    # Ginecológicos (Resolve o problema de útero/ovário)
    if "utero" in texto_completo or "útero" in texto_completo or "uterin" in texto_completo: return "Tumores Ginecológicos"
    if "ovario" in texto_completo or "ovário" in texto_completo: return "Tumores Ginecológicos"
    if "tuba" in texto_completo or "anexo" in texto_completo: return "Tumores Ginecológicos"
    if "vulva" in texto_completo or "vagina" in texto_completo: return "Tumores Ginecológicos"
    if "endometri" in texto_completo or "miometrio" in texto_completo: return "Tumores Ginecológicos"
    if "endocervical" in texto_completo or "ectocervical" in texto_completo: return "Tumores Ginecológicos"
    if "colo" in texto_completo and ("uterin" in texto_completo or "utero" in texto_completo): return "Tumores Ginecológicos"

    # Mama
    if "mama" in texto_completo: return "Tumores de Mama"
    if "mastectomia" in texto_completo: return "Tumores de Mama"

    # SNC
    if "cerebro" in texto_completo or "cérebro" in texto_completo or "snc" in texto_completo: return "Tumores do Sistema Nervoso Central"

    # GRUPO 2: DIGESTIVO
    
    # Colorretais - ATENÇÃO COM A PALAVRA "RETO"
    if "colon" in texto_completo or "cólon" in texto_completo or "sigmoide" in texto_completo or 'pericólico' in texto_completo or 'Ceco' in texto_completo or 'cecal' in texto_completo: 
        return "Tumores colorretais"
    
    if "reto" in texto_completo or "retal" in texto_completo:
        # Ignora se for "transretal" (comum em próstata) ou "retovaginal" (ginecológico)
        # Como já foi checado próstata e gineco acima, a chance de erro diminui, 
        if "transretal" not in texto_completo and "vagin" not in texto_completo:
            return "Tumores colorretais"
    
    if "anal" in texto_completo or "anus" in texto_completo or "ânus" in texto_completo:
        return "Tumores colorretais"

    # Digestivo Alto
    if "estomago" in texto_completo or "gastric" in texto_completo or "gástrico" in texto_completo: return "Tumores do Aparelho Digestivo Alto"
    if "esofago" in texto_completo or "esôfago" in texto_completo: return "Tumores do Aparelho Digestivo Alto"
    if "pancreas" in texto_completo or "pâncreas" in texto_completo: return "Tumores do Aparelho Digestivo Alto"
    if "figado" in texto_completo or "fígado" in texto_completo or "hepat" in texto_completo: return "Tumores do Aparelho Digestivo Alto"
    if "biliar" in texto_completo or "colédoco" in texto_completo or "bile" in texto_completo: return "Tumores do Aparelho Digestivo Alto"

    # TÓRAX
    if "pulmao" in texto_completo or "pulmão" in texto_completo or "bronqui" in texto_completo: return "Tumores de Pulmão e Tórax"
    if "mediastino" in texto_completo or "pleura" in texto_completo or "timo" in texto_completo: return "Tumores de Pulmão e Tórax"
    
    # PESCOÇO
    if "tireoide" in texto_completo or "laringe" in texto_completo or "faringe" in texto_completo: return "Tumores de cabeça e pescoço"
    if "boca" in texto_completo or "lingua" in texto_completo or "língua" in texto_completo: return "Tumores de cabeça e pescoço"
    if "salivar" in texto_completo or "parotida" in texto_completo: return "Tumores de cabeça e pescoço"
    if "linfonodo cervical" in texto_completo: return "Tumores de cabeça e pescoço"
    if "glote" in texto_completo: return "Tumores de cabeça e pescoço"
    if "gengiva" in texto_completo or "palato" in texto_completo or "Amígdala" in texto_completo: return "Tumores de cabeça e pescoço"
    if "Carcinoma Adenoide Cístico" in texto_completo: return "Tumores de cabeça e pescoço"

    # REGRA: Busca no Mapa do Arquivo TXT
    best_match_cr = "INDET"
    longest_match_len = 0
    for key, cr in malignant_map_rules.items():
        if key in diag_lower:
            # Lógica extra: Se o mapa diz Colorretal, mas o texto não tem colon/reto/anal, ignoramos
            # Isso evita falsos positivos de adenocarcinoma genérico
            if cr == "Tumores colorretais":
                termos_colorretais = ["colon", "cólon", "reto", "retal", "sigmoide", "anal", "ânus", "ceco", "cecal", "retossigmoide"]
                tem_termo_anat = any(t in texto_completo for t in termos_colorretais)
                if not tem_termo_anat:
                    continue # Pula essa correspondência do mapa se não tiver anatomia compatível
            
            if len(key) > longest_match_len:
                longest_match_len = len(key)
                best_match_cr = cr
    
    if best_match_cr != "INDET":
        return best_match_cr
    
    # REGRA: Ajustes Finais (casos não pegos acima)
    if "colon" in texto_completo or "cólon" in texto_completo or "sigmoide" in texto_completo: return "Tumores colorretais"
    if "reto" in texto_completo or "retal" in texto_completo:
        if "transretal" not in texto_completo: return "Tumores colorretais"
    if "anal" in texto_completo or "anus" in texto_completo: return "Tumores colorretais"

    # Digestivo Alto
    if "estomago" in texto_completo or "gástrico" in texto_completo: return "Tumores do Aparelho Digestivo Alto"
    if "esofago" in texto_completo or "esôfago" in texto_completo: return "Tumores do Aparelho Digestivo Alto"
    if "pancreas" in texto_completo or "pâncreas" in texto_completo: return "Tumores do Aparelho Digestivo Alto"
    if "figado" in texto_completo or "fígado" in texto_completo: return "Tumores do Aparelho Digestivo Alto"
    if "biliar" in texto_completo: return "Tumores do Aparelho Digestivo Alto"

    # Tórax
    if "pulmao" in texto_completo or "pulmão" in texto_completo: return "Tumores de Pulmão e Tórax"
    if "pleura" in texto_completo or "mediastino" in texto_completo: return "Tumores de Pulmão e Tórax"

    return "INDET"

In [3]:
# ========================================================================================
# --- 1. CARREGAMENTO ---
# ========================================================================================
# ========================================================================================
# --- 1. FUNÇÕES DE CARREGAMENTO ---
# ========================================================================================

def load_txt_knowledge_base(txt_path):
    """Carrega e processa o arquivo TXT linha a linha com metadados."""
    documents = []
    current_cr_key = "Desconhecido"
    
    try:
        with open(txt_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            line = line.strip()
            if not line: continue
            
            if line.startswith('###'):
                current_cr_key = line.replace('###', '').strip()
            elif line.startswith('-'):
                diagnosis_text = line.replace('-', '').strip()
                
                # Descobre o CR legível usando o dicionário global
                cr_real = header_to_cr.get(current_cr_key, "CR Desconhecido")
                
                # Cria texto rico para a LLM
                content = (
                    f"Diagnóstico: {diagnosis_text}. "
                    f"Grupo Técnico: {current_cr_key}. "
                    f"Centro de Referência Oficial: {cr_real}."
                )
                
                doc = Document(
                    page_content=content, 
                    metadata={
                        "source": txt_path,
                        "grupo": current_cr_key,
                        "cr": cr_real,
                        "diagnostico": diagnosis_text
                    }
                )
                documents.append(doc)
    except Exception as e:
        print(f"Erro ao carregar TXT: {e}")
            
    return documents

def load_structured_files_as_documents(filepath):
    """Carrega Excel ou CSV e converte em Documentos LangChain."""
    documents_list = []
    
    # --- LÓGICA EXCEL ---
    if filepath.endswith('.xlsx'):
        try:
            df = pd.read_excel(filepath)
            for index, row in df.iterrows():
                # Validação básica para evitar erros de índice
                if len(row) < 3: continue 

                cid = str(row.iloc[0]).strip()
                desc = str(row.iloc[1]).strip()
                cr = str(row.iloc[2]).strip()
                
                if cr.startswith("CR "): cr = cr[3:]
                
                if cr and cr != 'nan' and desc != 'nan':
                    content = f"O código CID {cid} referente ao diagnóstico '{desc}' é classificado no grupo: {cr}."
                    
                    doc = Document(
                        page_content=content,
                        metadata={"source": filepath, "row": index, "type": "excel_cid"}
                    )
                    documents_list.append(doc)
            print(f"Excel carregado: {len(documents_list)} documentos.")
            
        except Exception as e:
            print(f"Erro ao converter Excel para docs: {e}")

    # --- LÓGICA CSV ---
    elif filepath.endswith('.csv'):
        try:
            try: df = pd.read_csv(filepath)
            except: df = pd.read_csv(filepath, sep=';')
            
            for index, row in df.iterrows():
                # Validação básica
                if len(row) < 6: continue

                snomed = str(row.iloc[0]).strip()
                label = str(row.iloc[1]).strip()
                cid = str(row.iloc[3]).strip()
                desc_cid = str(row.iloc[4]).strip()
                cr = str(row.iloc[5]).strip()
                
                if pd.isna(row.iloc[5]) or cr == '': continue 
                if cr.startswith("CR "): cr = cr[3:]

                content = (f"Diagnóstico: {label}. Código SNOMED: {snomed}. "
                           f"Código CID: {cid} ({desc_cid}). "
                           f"Classificação correta: {cr}.")
                
                doc = Document(
                    page_content=content,
                    metadata={"source": filepath, "row": index, "type": "csv_snomed"}
                )
                documents_list.append(doc)
            print(f"CSV carregado. Total acumulado no batch: {len(documents_list)} documentos.")
            
        except Exception as e:
            print(f"Erro ao converter CSV para docs: {e}")
            
    return documents_list

# ========================================================================================
# --- 2. EXECUÇÃO DO CARREGAMENTO ---
# ========================================================================================

print("Iniciando carregamento dos documentos...")

# 1. Carrega o TXT usando sua função personalizada (Melhor que TextLoader)
docs_txt = load_txt_knowledge_base("diagnosticos_topografia.txt")

# 2. Carrega o Excel
docs_xlsx = load_structured_files_as_documents("Base CID_Classificação CR.xlsx")

# 3. Carrega o CSV
docs_csv = load_structured_files_as_documents("SNOMED_to_CID_2.csv")

# 4. JUNTA TUDO NA LISTA FINAL 'docs'
docs = docs_txt + docs_xlsx + docs_csv

print(f"Total de documentos prontos para indexação: {len(docs)}")

# ========================================================================================
# --- 3. INDEXAÇÃO VETORIAL E DEFINIÇÃO DO MODELO ---
# ========================================================================================

print("Criando índice vetorial local (isso pode demorar um pouquinho na 1ª vez)...")

embeddings = OllamaEmbeddings(model="nomic-embed-text") 

persist_dir = "./chroma_db_local"

if os.path.exists(persist_dir) and os.listdir(persist_dir):
    vectorstore = Chroma(persist_directory=persist_dir, embedding_function=embeddings)
else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_dir
    )
    
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 20}
)

# --- LLM LOCAL (GEMMA2) ---
llm = ChatOllama(
    model="gemma3:12b", 
    temperature=0,
    num_ctx=4096 
)

print("Sistema RAG inicializado com sucesso!")
# ========================================================================================
# --- 3. Definição do Prompt Estruturado ---
# ========================================================================================

PROMPT_SISTEMA = """
# PERSONA
Você é um especialista em codificação médica, treinado para mapear diagnósticos oncológicos (combinação de morfologia SNOMED e topografia) para os códigos de classificação internacional de doenças (CID-10).
 
# OBJETIVO
Sua única tarefa é analisar uma morfologia (DIAGNÓSTICO e SNOMED), uma topografia anatômica (TOPOGRAFIA) e um procedimento (PROCEDIMENTO) para encontrar todos os códigos CID correspondentes em uma lista de CIDS de referência fornecida no CONTEXTO.
 
# REGRAS
1) Caso a topografia seja "Nenhuma_topografia_encontrada" ou "Muitas_topografias_possiveis", não leve essa em consideração.
2) Mesmo que identifique que o conjunto é BENIGNO, retorne um CID ou mais CIDs aplicáveis.
3) Pode haver mais de um CID aplicável. Nesse caso, retorne todos os CIDs aplicáveis.
4) Busque no CONTEXTO um ou mais CIDs que representem a associação entre SNOMED, DIAGNÓSTICO e TOPOGRAFIA. Se não houver correspondência direta, use seu conhecimento próprio para inferir um ou mais CIDs  prováveis para a combinação de SNOMED, DIAGNÓSTICO e TOPOGRAFIA dada, mas apenas se o CID estiver listado no CONTEXTO.
5) Se você concluir que nenhum CID do CONTEXTO corresponde à combinação de SNOMED, DIAGNÓSTICO e TOPOGRAFIA, responda com a string `Nenhum_CID_encontrado`.
6) Sua resposta deve conter **APENAS** os códigos CID encontrados, separados por ponto e vírgula (`;`), sem espaços adicionais. Não inclua cabeçalhos, explicações ou qualquer texto adicional.
 
# EXEMPLOS
## Exemplo 1: Múltiplas correspondências
- DADOS: SNOMED: M-82003, DIAGNÓSTICO: "Carcinoma, duto salivar", TOPOGRAFIA: "Glândula salivar"
- CONTEXTO: Contém "- C08: Neoplasia maligna de outras glândulas salivares maiores e as não especificadas", "- C088: Neoplasia maligna das glândulas salivares maiores com lesão invasiva" e "- C089: Neoplasia maligna da glândula salivar maior, não especificada"
- RESPOSTA ESPERADA: C08;C088;C089
 
## Exemplo 2: Correspondência Única
- DADOS: SNOMED: M-90603, DIAGNÓSTICO: "Disgerminoma", TOPOGRAFIA: "Ovário"
- CONTEXTO: Contém "- C56: Neoplasia maligna do ovário"
- RESPOSTA ESPERADA: C56
"""
 
PROMPT_USUARIO = """
<CONTEXTO_RECUPERADO>
{contexto}
</CONTEXTO_RECUPERADO>

<DADOS_PARA_ANALISE>
  <SNOMED>{snomed}</SNOMED>
  <DIAGNOSTICO>{label}</DIAGNOSTICO>
  <TOPOGRAFIA>{topografia}</TOPOGRAFIA>
  <PROCEDIMENTO>{procedimento}</PROCEDIMENTO>
</DADOS_PARA_ANALISE>

Qual a resposta?
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", PROMPT_SISTEMA),
    ("human", PROMPT_USUARIO)
])

rag_chain = prompt_template | llm | StrOutputParser()

# ==============================================================================
# 4. EXECUÇÃO OTIMIZADA (DEDUPLICAÇÃO + HÍBRIDO)
# ==============================================================================

print("Carregando mapas de regras (Dicionários)...")

# 1. Carrega TXT
mapa_v1 = parse_knowledge_base_file("diagnosticos_topografia.txt")

# 2. Carrega Excel (Atenção ao nome exato do arquivo)
#mapa_v2 = parse_knowledge_base_file("   .xlsx")

# 3. Carrega CSV
mapa_v3 = parse_knowledge_base_file("SNOMED_to_CID_2.csv")

# 4. Funde tudo num único dicionário mestre
# O Python prioriza o último (v3 sobrescreve v2, que sobrescreve v1)
malignant_map_rules = {**mapa_v1, **mapa_v3}

print(f"Mapa de regras carregado com {len(malignant_map_rules)} termos únicos.")


nome_csv_entrada = 'SNOMED_to_CID_2.csv' # Ajuste o nome se necessário
nome_checkpoint = 'checkpoint_classificacao.csv'

print(f"3. Lendo arquivo de dados: {nome_csv_entrada}")
df_original = pd.read_csv(nome_csv_entrada, sep=';') # Ajuste sep=';' se necessário

# 1. Mapeamento de Colunas (De-Para)
# Se o arquivo novo tiver nomes diferentes (ex: "Diagnostico" em vez de "DesDiagnostico")
# Ajuste este dicionário conforme o cabeçalho do seu arquivo novo
mapa_colunas = {
    'Diagnóstico': 'Diagnóstico',  # Ex: Nome no arquivo novo -> Nome que o código usa
    'Topografia': 'Topografia',
    'SNOMED': 'SNOMED'
    # Adicione outros se necessário
}
df_original.rename(columns=mapa_colunas, inplace=True)

# 2. Tratamento da Coluna Faltante (Procedimento)
# Se a coluna 'DesProcedimento' não existir, criamos ela preenchida com texto vazio
if 'DesProcedimento' not in df_original.columns:
    print("AVISO: Coluna 'DesProcedimento' não encontrada. Criando com valor padrão.")
    df_original['DesProcedimento'] = "Não informado"

# Deduplicação
# Adicionando SNOMED e DesProcedimento na criação da chave
df_original['chave_unica'] = (
    df_original['Diagnóstico'].astype(str) + " | " + 
    df_original['Topografia'].astype(str) + " | " + 
    df_original['SNOMED'].astype(str) + " | " + 
    df_original['DesProcedimento'].astype(str)
)

# incluindo as colunas no df_unicos
df_unicos = df_original[['chave_unica', 'Diagnóstico', 'Topografia', 'SNOMED', 'DesProcedimento']].drop_duplicates(subset=['chave_unica']).copy()

total_unicos = len(df_unicos)
print(f"-> Otimização: {len(df_original)} linhas reduzidas para {total_unicos} casos únicos.")

# Carregando checkpoint se existir - retomando processamento - precisa deletar se quiser reprocessar tudo utilizando outro modelo ou método
resultados_processados = []
casos_ja_feitos = set()
if os.path.exists(nome_checkpoint):
    try:
        df_check = pd.read_csv(nome_checkpoint, sep=';')
        resultados_processados = df_check.to_dict('records')
        casos_ja_feitos = set(df_check['chave_unica'].values)
        print(f"-> Retomando de checkpoint: {len(casos_ja_feitos)} já processados.")
    except:
        print("-> Checkpoint ilegível, iniciando do zero.")

print("4. Iniciando processamento híbrido (modo paralelo)...")

batch_save = 20

# ======================================================
# FUNÇÃO QUE PROCESSA UMA LINHA (ANTES ERA O LOOP)
# ======================================================

def process_row(row_dict):

    chave = row_dict['chave_unica']

    if chave in casos_ja_feitos:
        return None

    diag = str(row_dict['Diagnóstico'])
    topo = str(row_dict['Topografia'])
    snomed = str(row_dict['SNOMED'])
    procedimento = str(row_dict.get('DesProcedimento', 'Não informado'))

    # -------- PASSO A: STATUS POR REGRA --------
    status_regra = classify_case(diag, topo, snomed, malignant_map_rules)

    # -------- PASSO B: RAG (somente quando INDET) --------
    metodo = "REGRA"
    cid_final = "Nenhum_CID_encontrado"
    context_text = ""

    if status_regra == "INDET":

        metodo = "RAG"

        topo_llm = topo
        if topo in ["Nenhuma_topografia_encontrada", "Muitas_topografias_possiveis"]:
            topo_llm = ""

        try:
            query = f"Diagnóstico: {diag} Topografia: {topo_llm} Código: {snomed}"
            docs_recuperados = retriever.invoke(query)

            # reduz tamanho do contexto → melhora velocidade
            context_text = "\n\n".join([d.page_content[:250] for d in docs_recuperados])

            response = rag_chain.invoke({
                "contexto": context_text,
                "snomed": snomed,
                "label": diag,
                "topografia": topo_llm,
                "procedimento": procedimento
            })

            cid_final = response.strip().split('\n')[0]
            cid_final = cid_final.replace(".", "").replace('"', '').strip()

            if not cid_final:
                cid_final = "Nenhum_CID_encontrado"

            # -------- FILTRO DE CID (regra #4 do prompt) --------
            allowed = extract_cids(context_text)

            if cid_final != "Nenhum_CID_encontrado":
                predicted = [c.strip().replace(".", "") for c in cid_final.split(";") if c.strip()]
                predicted = [c for c in predicted if c in allowed]
                cid_final = ";".join(predicted) if predicted else "Nenhum_CID_encontrado"

        except Exception as e:
            cid_final = "ERRO_PROCESSAMENTO"
            metodo = "ERRO_RAG"
            print(f"Erro IA no caso {chave}: {e}")

    # -------- RETORNO DA THREAD --------
    return {
        "chave_unica": chave,
        "CID_Preditos": cid_final,
        "Metodo_Utilizado": metodo,
        "Status_Regra": status_regra
    }


# ======================================================
# EXECUÇÃO PARALELA
# ======================================================

rows = df_unicos.to_dict("records")

with ThreadPoolExecutor(max_workers=3) as executor:

    for resultado in tqdm(executor.map(process_row, rows), total=len(rows)):

        if resultado is None:
            continue

        resultados_processados.append(resultado)

        # checkpoint periódico
        if len(resultados_processados) % batch_save == 0:
            pd.DataFrame(resultados_processados).to_csv(
                nome_checkpoint, sep=';', index=False, encoding='utf-8-sig'
            )

    # Checkpoint periódico
    if len(resultados_processados) % batch_save == 0:
        pd.DataFrame(resultados_processados).to_csv(nome_checkpoint, sep=';', index=False, encoding='utf-8-sig')

# Salva lista de únicos final
df_resultados_unicos = pd.DataFrame(resultados_processados)
df_resultados_unicos.to_csv(nome_checkpoint, sep=';', index=False, encoding='utf-8-sig')


Iniciando carregamento dos documentos...
Excel carregado: 1043 documentos.
CSV carregado. Total acumulado no batch: 5505 documentos.
Total de documentos prontos para indexação: 7910
Criando índice vetorial local (isso pode demorar um pouquinho na 1ª vez)...


C:\Users\rafae\AppData\Local\Temp\ipykernel_39748\4168188363.py:148: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=persist_dir, embedding_function=embeddings)


Sistema RAG inicializado com sucesso!
Carregando mapas de regras (Dicionários)...
CSV processado: Mapa agora tem 4158 termos totais.
Mapa de regras carregado com 4949 termos únicos.
3. Lendo arquivo de dados: SNOMED_to_CID_2.csv
AVISO: Coluna 'DesProcedimento' não encontrada. Criando com valor padrão.
-> Otimização: 9444 linhas reduzidas para 4601 casos únicos.
-> Retomando de checkpoint: 12300 já processados.
4. Iniciando processamento híbrido (modo paralelo)...


100%|██████████| 4601/4601 [05:06<00:00, 15.00it/s] 


In [4]:
# ==============================================================================
# 5. MERGE FINAL E SALVAMENTO
# ==============================================================================
print("5. Gerando arquivo final completo...")

df_final = pd.merge(df_original, df_resultados_unicos, on='chave_unica', how='left')

cols = ['SNOMED', 'Diagnóstico', 'Topografia', 'DesProcedimento', 'CID_Preditos', 'Metodo_Utilizado', 'Status_Regra']
cols += [c for c in df_final.columns if c not in cols and c != 'chave_unica']
df_final = df_final[cols]

df_final.to_csv("CID_paciente_CLASSIFICADO_FINAL.csv", sep=';', index=False, encoding='utf-8-sig')
print("SUCESSO! Arquivo 'CID_paciente_CLASSIFICADO_FINAL.csv' gerado.")


5. Gerando arquivo final completo...
SUCESSO! Arquivo 'CID_paciente_CLASSIFICADO_FINAL.csv' gerado.


In [ ]:
# ==============================================================================
# 6. ANÁLISE DE RESULTADOS
# ==============================================================================


plt.figure(figsize=(10, 6))
counts_metodo = df_final['Metodo_Utilizado'].value_counts()

# Gráfico de Pizza
plt.pie(counts_metodo, labels=counts_metodo.index, autopct='%1.1f%%', startangle=140, colors=['#66b3ff','#99ff99','#ffcc99'])
plt.title('Eficiência da Triagem: Regras Determinísticas vs Inteligência Artificial')
# plt.savefig("relatorios_img/kpi_eficiencia_automacao.png")
# plt.close()

print(f"-> Gráfico de Eficiência salvo em relatorios_img")

NameError: name 'df_final' is not defined

<Figure size 1000x600 with 0 Axes>